# Insider Trading Scanner — token-free (Colab)

Pulls real insider (Form 4) transactions **for free** from OpenInsider (no key, no token). Built around the
vault verdict: **read the BREADTH, not the DOLLARS.** A '$15.6B insiders selling!' headline is mostly the
denominator — megacap comp is equity, stocks are up thousands of %, and most sales are pre-scheduled
10b5-1. The signal isn't sell-dollars; it's **buying breadth** (insiders buy for one reason, sell for a
thousand). So this tool leads with **cluster buys** (multiple insiders buying the SAME name = the real
tell) and gives a clean **buy-vs-sell tally per watchlist name** so you see conviction, not headlines.

Run top-to-bottom. If OpenInsider is slow/blocks, it says so and skips — not fatal.


In [ ]:
import sys, subprocess
def _pip(p): subprocess.run([sys.executable,'-m','pip','install','-q',p])
try:
    import pandas as pd, requests, lxml
except Exception:
    _pip('pandas'); _pip('requests'); _pip('lxml'); import pandas as pd, requests
import io, re
from datetime import datetime, timezone, timedelta

UA = {'User-Agent':'Mozilla/5.0 (research; personal use)'}
def fetch_tables(url):
    r = requests.get(url, headers=UA, timeout=30); r.raise_for_status()
    return pd.read_html(io.StringIO(r.text))

def pick_insider_table(tables):
    for t in tables:
        cols = [str(c) for c in t.columns]
        if any('Ticker' in c for c in cols) and any(('Trade' in c) or ('Insider' in c) for c in cols):
            return t
    return None

def norm_cols(df):
    df = df.copy(); df.columns = [re.sub(r'\s+',' ',str(c)).strip() for c in df.columns]
    return df

def col(df, *names):
    for n in names:
        for c in df.columns:
            if n.lower() in c.lower(): return c
    return None

def parse_money(x):
    s = re.sub(r'[^0-9.\-]','', str(x))
    try: return float(s)
    except: return 0.0

def parse_date(x):
    for fmt in ('%Y-%m-%d','%Y-%m-%d %H:%M:%S'):
        try: return datetime.strptime(str(x)[:19].strip(), fmt).replace(tzinfo=timezone.utc)
        except: pass
    return None


## CONFIG


In [ ]:
WATCHLIST = ['NVDA','PLTR','META','MU','AAPL','MSFT','GOOGL','AMZN','TSM','AVGO','NFLX','LLY','NOW','ORCL']
DAYS = 120   # look-back window for the per-ticker tally


## Panel A — latest CLUSTER BUYS (the breadth signal that actually matters)
2+ insiders buying the SAME stock in a tight window = high-conviction. This is the one to read.


In [ ]:
print('='*74); print('  LATEST CLUSTER BUYS (multiple insiders, same name = conviction)'); print('='*74)
try:
    tabs = fetch_tables('http://openinsider.com/latest-cluster-buys')
    t = norm_cols(pick_insider_table(tabs))
    tk, tn, tt, val, td = (col(t,'Ticker'), col(t,'Company','Insider'), col(t,'Trade Date','Trade'),
                            col(t,'Value'), col(t,'Trade Date'))
    show = t[[c for c in [td,tk,col(t,'Company'),val,col(t,'Qty'),col(t,'Owned')] if c]].head(20)
    print(show.to_string(index=False))
except Exception as e:
    print('  cluster-buys pull failed (site slow/blocked?):', e)


## Panel B — per-watchlist BUY vs SELL tally (count + $, last N days)


In [ ]:
now = datetime.now(timezone.utc); cutoff = now - timedelta(days=DAYS)
rows=[]
for tk in WATCHLIST:
    try:
        t = pick_insider_table(fetch_tables('http://openinsider.com/'+tk))
        if t is None: rows.append(dict(ticker=tk, note='no table')); continue
        t = norm_cols(t)
        c_type, c_val, c_date, c_name = col(t,'Trade Type','Trade'), col(t,'Value'), col(t,'Trade Date'), col(t,'Insider Name','Insider')
        buys_n=sells_n=0; buys_v=sells_v=0.0; buyers=set(); sellers=set()
        for _,r in t.iterrows():
            d = parse_date(r.get(c_date)) if c_date else None
            if d and d < cutoff: continue
            ty = str(r.get(c_type,'')).strip().upper(); v = abs(parse_money(r.get(c_val)))
            nm = str(r.get(c_name,'')).strip()
            if ty.startswith('P'): buys_n+=1; buys_v+=v; buyers.add(nm)
            elif ty.startswith('S'): sells_n+=1; sells_v+=v; sellers.add(nm)
        rows.append(dict(ticker=tk, buys=buys_n, buyers=len(buyers), buy_$M=round(buys_v/1e6,1),
                         sells=sells_n, sellers=len(sellers), sell_$M=round(sells_v/1e6,1)))
    except Exception as e:
        rows.append(dict(ticker=tk, note=str(e)[:40]))
df = pd.DataFrame(rows).set_index('ticker')
print(f'INSIDER BUY vs SELL — last {DAYS} days (count = breadth; $M = the noisy denominator)')
print(df.to_string())
print('\nRead: a name with ANY open-market BUYS (esp. multiple buyers) is the signal.')
print('Big sell_$M with 0 buys at a megacap = mostly 10b5-1 / option exercises = noise.')


## Panel C — market pulse: recent buys vs sells breadth


In [ ]:
try:
    buys = norm_cols(pick_insider_table(fetch_tables('http://openinsider.com/latest-insider-purchases-25k')))
    sells= norm_cols(pick_insider_table(fetch_tables('http://openinsider.com/latest-insider-sales')))
    def uniq_tickers(t): c=col(t,'Ticker'); return t[c].nunique() if c else len(t)
    print('Recent open-market PURCHASE filings (25k+):', len(buys), '| unique tickers:', uniq_tickers(buys))
    print('Recent SALE filings:', len(sells), '| unique tickers:', uniq_tickers(sells))
    print('\nNOTE: sellers ALWAYS outnumber buyers (equity comp). Watch the BUY side trend/cluster count,',
          'not the raw ratio.')
except Exception as e:
    print('  market-pulse pull failed:', e)


### How to read / retune
- **Panel A (cluster buys)** is the highest-value screen — insiders rarely buy, and *several* buying one
  name is the strongest insider signal there is. Empty is normal; a familiar ticker appearing is notable.
- **Panel B**: `buyers`/`sellers` are unique-person COUNTS (breadth); `$M` is the dollar figure that the
  headlines inflate. A megacap showing `buys 0 / sell_$M 3300` is the '$3.3B Nvidia selling' headline —
  and it's noise (10b5-1 + a stock up ~2000%). A small-cap showing `buyers 4 / buy_$M 2` is real conviction.
- Change `WATCHLIST` / `DAYS` in CONFIG. Add tickers freely.
- Paste any panel back to me to ingest. Source: openinsider.com (free Form 4 aggregator).
